# 01 - Inicialização do Projeto

**Projeto:** Análise do Mercado Imobiliário Português  
**Fase CRISP-DM:** Preparação inicial do projeto e dos dados  
**Objetivo:** carregar o dataset original, aplicar uma limpeza inicial conservadora e criar uma primeira versão processada para as fases seguintes.  
**Última alteração:** 16/06/2026

Este notebook estabelece a base técnica do projeto. A prioridade é garantir que os dados originais permanecem intactos, que os caminhos funcionam em ambiente local e no Kaggle, e que as transformações iniciais são simples, explícitas e reproduzíveis.

Entradas esperadas:
- `data/raw/portugal_listinigs.csv`, quando o ficheiro existe localmente.
- Dataset carregado no Kaggle, quando disponível em `/kaggle/input`.
- Ficheiro raw no GitHub, usado como alternativa quando as fontes anteriores não existem.

Saída esperada:
- `data/processed/portugal_listings_initial_clean.csv`, em ambiente local.
- `/kaggle/working/portugal_listings_initial_clean.csv`, em ambiente Kaggle.


## Âmbito

Tarefas incluídas:

- configurar caminhos relativos do projeto;
- carregar o ficheiro original sem o alterar;
- normalizar nomes de colunas para `snake_case`;
- converter tipos de dados principais;
- criar variáveis derivadas iniciais;
- preparar uma versão processada para uso posterior.

Tarefas fora deste notebook:

- análise exploratória detalhada;
- tratamento final de valores extremos;
- imputação de valores em falta;
- modelação preditiva.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


In [2]:
GITHUB_BASE_URL = "https://raw.githubusercontent.com/LuiscnFigueira/portugal-real-estate-market-analysis/main"
DATASET_FILENAME = "portugal_listinigs.csv"
PROCESSED_FILENAME = "portugal_listings_initial_clean.csv"
GITHUB_RAW_DATA_URL = f"{GITHUB_BASE_URL}/data/raw/{DATASET_FILENAME}"
GITHUB_PROCESSED_DATA_URL = f"{GITHUB_BASE_URL}/data/processed/{PROCESSED_FILENAME}"


def find_project_root(start=None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        markers = [candidate / "README.md", candidate / "requirements.txt", candidate / "data"]
        if all(marker.exists() for marker in markers):
            return candidate
    return Path("/kaggle/working") if Path("/kaggle/working").exists() else current


def find_raw_data_source(project_root: Path):
    local_candidates = [
        project_root / "data" / "raw" / DATASET_FILENAME,
        (Path.cwd() / ".." / "data" / "raw" / DATASET_FILENAME).resolve(),
    ]

    for path in local_candidates:
        if path.exists():
            return path

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        matches = sorted(kaggle_input.rglob(DATASET_FILENAME))
        if matches:
            return matches[0]

    return GITHUB_RAW_DATA_URL


def processed_data_path(project_root: Path) -> Path:
    if Path("/kaggle/working").exists():
        return Path("/kaggle/working") / PROCESSED_FILENAME
    return project_root / "data" / "processed" / PROCESSED_FILENAME


PROJECT_ROOT = find_project_root()
RAW_DATA_PATH = find_raw_data_source(PROJECT_ROOT)
PROCESSED_DATA_PATH = processed_data_path(PROJECT_ROOT)

print(f"Raiz de trabalho: {PROJECT_ROOT}")
print(f"Dataset original: {RAW_DATA_PATH}")
print(f"Dataset processado: {PROCESSED_DATA_PATH}")


Raiz de trabalho: /kaggle/working
Dataset original: /kaggle/input/datasets/luvathoms/portugal-real-estate-2024/portugal_listinigs.csv
Dataset processado: /kaggle/working/portugal_listings_initial_clean.csv


## Carregamento dos Dados

O ficheiro em `data/raw/` é tratado como fonte original e não deve ser editado manualmente. Todas as transformações são feitas numa cópia em memória.


In [3]:
df_raw = pd.read_csv(RAW_DATA_PATH, low_memory=False)
df = df_raw.copy()

print(f"Dataset bruto: {df_raw.shape[0]:,} linhas e {df_raw.shape[1]} colunas")
display(df_raw.head())


Dataset bruto: 135,536 linhas e 25 colunas


,Price,District,City,Town,Type,EnergyCertificate,GrossArea,TotalArea,Parking,HasParking,...,Elevator,ElectricCarsCharging,TotalRooms,NumberOfBedrooms,NumberOfWC,ConservationStatus,LivingArea,LotSize,BuiltArea,NumberOfBathrooms
0,780000.0,Vila Real,Valpaços,Carrazedo de Montenegro e Curros,Farm,NC,200.0,552450.0,0.0,False,...,False,NaN,NaN,NaN,NaN,NaN,120.0,NaN,NaN,0.0
1,223000.0,Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,NaN,81.0,1.0,True,...,True,NaN,2.0,NaN,NaN,NaN,81.0,NaN,NaN,2.0
2,228000.0,Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,NaN,108.0,1.0,True,...,True,NaN,2.0,NaN,NaN,NaN,108.0,NaN,NaN,2.0
3,250000.0,Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,NaN,114.0,1.0,True,...,True,NaN,2.0,NaN,NaN,NaN,114.0,NaN,NaN,0.0
4,250000.0,Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,NaN,114.0,1.0,True,...,True,NaN,2.0,NaN,NaN,NaN,114.0,NaN,NaN,2.0


## Normalização de Colunas

Os nomes originais são convertidos para `snake_case` para facilitar leitura, reutilização em código e documentação técnica.


In [4]:
column_names = {
    "Price": "price",
    "District": "district",
    "City": "city",
    "Town": "town",
    "Type": "type",
    "EnergyCertificate": "energy_certificate",
    "GrossArea": "gross_area",
    "TotalArea": "total_area",
    "Parking": "parking",
    "HasParking": "has_parking",
    "Floor": "floor",
    "ConstructionYear": "construction_year",
    "EnergyEfficiencyLevel": "energy_efficiency_level",
    "PublishDate": "publish_date",
    "Garage": "garage",
    "Elevator": "elevator",
    "ElectricCarsCharging": "electric_cars_charging",
    "TotalRooms": "total_rooms",
    "NumberOfBedrooms": "number_of_bedrooms",
    "NumberOfWC": "number_of_wc",
    "ConservationStatus": "conservation_status",
    "LivingArea": "living_area",
    "LotSize": "lot_size",
    "BuiltArea": "built_area",
    "NumberOfBathrooms": "number_of_bathrooms",
}

df = df.rename(columns=column_names)

display(pd.DataFrame({"colunas_normalizadas": df.columns}))


,colunas_normalizadas
0,price
1,district
2,city
3,town
4,type
5,energy_certificate
6,gross_area
7,total_area
8,parking
9,has_parking


## Conversão de Tipos

As conversões abaixo são preliminares e preservam valores ausentes sempre que possível. Variáveis inteiras com valores em falta usam o tipo `Int64` do pandas.


In [5]:
df["publish_date"] = pd.to_datetime(df["publish_date"], errors="coerce")

continuous_cols = [
    "price",
    "gross_area",
    "total_area",
    "living_area",
    "lot_size",
    "built_area",
]

for col in continuous_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

boolean_cols = [
    "has_parking",
    "garage",
    "elevator",
    "electric_cars_charging",
]

boolean_map = {
    "true": True,
    "false": False,
    "yes": True,
    "no": False,
    "sim": True,
    "não": False,
    "nao": False,
    "nan": pd.NA,
    "<na>": pd.NA,
}

for col in boolean_cols:
    df[col] = (
        df[col]
        .astype("string")
        .str.lower()
        .str.strip()
        .map(boolean_map)
        .astype("boolean")
    )

categorical_cols = [
    "district",
    "city",
    "town",
    "type",
    "energy_certificate",
    "floor",
    "energy_efficiency_level",
    "conservation_status",
]

for col in categorical_cols:
    df[col] = df[col].astype("category")

integer_cols = [
    "parking",
    "construction_year",
    "total_rooms",
    "number_of_bedrooms",
    "number_of_wc",
    "number_of_bathrooms",
]

for col in integer_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

display(pd.DataFrame({"tipo": df.dtypes.astype(str)}).reset_index(names="coluna"))


,coluna,tipo
0,price,float64
1,district,category
2,city,category
3,town,category
4,type,category
5,energy_certificate,category
6,gross_area,float64
7,total_area,float64
8,parking,Int64
9,has_parking,boolean


## Regras Conservadoras de Validação

Nesta fase são apenas marcados como ausentes valores objetivamente inválidos para o cálculo das variáveis derivadas. Decisões mais fortes sobre valores extremos devem ser tomadas na preparação final dos dados.


In [6]:
df.loc[df["price"] <= 0, "price"] = np.nan
df.loc[df["total_area"] <= 0, "total_area"] = np.nan

area_cols = ["gross_area", "living_area", "lot_size", "built_area"]
for col in area_cols:
    df.loc[df[col] <= 0, col] = np.nan

count_cols = ["number_of_wc", "number_of_bathrooms"]
for col in count_cols:
    df.loc[df[col] < 0, col] = np.nan

validation_summary = pd.Series({
    "price_em_falta": df["price"].isna().sum(),
    "total_area_em_falta": df["total_area"].isna().sum(),
    "gross_area_em_falta": df["gross_area"].isna().sum(),
    "living_area_em_falta": df["living_area"].isna().sum(),
    "lot_size_em_falta": df["lot_size"].isna().sum(),
    "built_area_em_falta": df["built_area"].isna().sum(),
})
display(validation_summary.to_frame("valor"))


,valor
price_em_falta,300
total_area_em_falta,9316
gross_area_em_falta,108229
living_area_em_falta,30984
lot_size_em_falta,96579
built_area_em_falta,109162


## Variáveis Derivadas

`price_m2` é útil para análise exploratória, mas não deve ser usada como variável explicativa num modelo que prevê `price`, porque deriva diretamente da variável alvo.


In [7]:
REFERENCE_YEAR = 2026

df["price_m2"] = df["price"] / df["total_area"]
df["price_m2"] = df["price_m2"].replace([np.inf, -np.inf], np.nan)
df.loc[df["price_m2"] <= 0, "price_m2"] = np.nan

df["property_age"] = REFERENCE_YEAR - df["construction_year"]
df.loc[df["property_age"] < 0, "property_age"] = np.nan

df["publish_year"] = df["publish_date"].dt.year.astype("Int64")
df["publish_month"] = df["publish_date"].dt.month.astype("Int64")

display(df[["price", "total_area", "price_m2", "property_age", "publish_year", "publish_month"]].head())


,price,total_area,price_m2,property_age,publish_year,publish_month
0,780000.0,552450.0,1.411892,<NA>,<NA>,<NA>
1,223000.0,81.0,2753.086420,<NA>,<NA>,<NA>
2,228000.0,108.0,2111.111111,<NA>,<NA>,<NA>
3,250000.0,114.0,2192.982456,<NA>,<NA>,<NA>
4,250000.0,114.0,2192.982456,<NA>,<NA>,<NA>


## Versão Processada Inicial

A versão processada remove duplicados exatos e registos sem `price`, porque `price` é a variável alvo. O ficheiro original permanece intacto.


In [8]:
df_processed = (
    df
    .drop_duplicates()
    .dropna(subset=["price"])
    .reset_index(drop=True)
)

processed_summary = pd.Series({
    "linhas_processadas": df_processed.shape[0],
    "colunas_processadas": df_processed.shape[1],
    "price_em_falta": df_processed["price"].isna().sum(),
    "duplicados_exatos": df_processed.duplicated().sum(),
    "total_area_nao_positiva": (df_processed["total_area"] <= 0).sum(),
    "price_m2_infinito": np.isinf(df_processed["price_m2"]).sum(),
})
display(processed_summary.to_frame("valor"))


,valor
linhas_processadas,126242
colunas_processadas,29
price_em_falta,0
duplicados_exatos,0
total_area_nao_positiva,0
price_m2_infinito,0


In [9]:
PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df_processed.to_csv(PROCESSED_DATA_PATH, index=False)

print(f"Dataset processado guardado em: {PROCESSED_DATA_PATH}")


Dataset processado guardado em: /kaggle/working/portugal_listings_initial_clean.csv


## Conclusão

Esta fase inicial preparou uma versão processada do dataset sem alterar os dados originais em `data/raw/`. Foram normalizados nomes de colunas, convertidos tipos de dados, aplicadas regras conservadoras de validação e criadas variáveis derivadas úteis para análise exploratória e modelação.

Artefactos produzidos:
- Dataset processado inicial em `data/processed/portugal_listings_initial_clean.csv`, ou em `/kaggle/working/` quando executado no Kaggle.
- Validações básicas sobre preço, áreas, duplicados e variáveis numéricas críticas.
- Base preparada para a fase de compreensão dos dados.

Limitações e cuidados:
- As transformações desta fase são deliberadamente conservadoras.
- Valores extremos não são removidos automaticamente; devem ser estudados na análise exploratória.
- As conclusões sobre fatores que influenciam o preço só devem ser feitas após validação estatística e modelação.

Próxima etapa:
- Avançar para o notebook `02_data_understanding.ipynb`, onde serão analisadas distribuições, valores em falta, duplicados, relações entre variáveis e primeiros padrões do mercado.

**Última alteração:** 16/06/2026
